# 08 eGeMAPS

The extended Geneva Minimalistic Acoustic Parameter Set v02 (Eyben et al., 2016): 25 low-level descriptors (LLDs) every 10 ms and 88 functionals per clip, with openSMILE's names and order, under AFX's MIT licence. This notebook runs it on synthetic vowels whose F0, jitter and shimmer are known, then uses the `egemaps` preset in a scikit-learn pipeline. Needs `pip install "afx-audio[notebooks]"`; the last cell compares with openSMILE when it is installed.

Implementation notes and the per-functional agreement with openSMILE are in `docs/egemaps.md`. Cite Eyben et al. (2016) with the preset id `egemaps@1`.

In [ ]:
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import afx_audio as afx
from afx_audio.sklearn import AFXFeatureExtractor
import demo_data

SR = 16000

## A synthetic voice

`demo_data.synthetic_voice` builds a sustained vowel: Rosenberg glottal pulses, three formant resonators, lip radiation and white noise. Each period is the nominal one times `1 + jitter * N(0, 1)` and each pulse amplitude `1 + shimmer * N(0, 1)`. With `return_truth=True` it also returns the realized mean F0 in semitones above 27.5 Hz, the local jitter (mean |T_i - T_i-1| / mean T) and the local shimmer (mean |20 log10(A_i / A_i-1)| in dB), the definitions eGeMAPS uses.

The clip below glides from 110 to 180 Hz between 0.25 s and 1.75 s, with silence around it.

In [ ]:
x, truth = demo_data.synthetic_voice(f0=(110.0, 180.0), jitter=0.01, shimmer=0.05, seconds=2.0, return_truth=True)
print(f'{x.size / SR:.1f} s;', {k: round(v, 4) for k, v in truth.items()})

## The 88 functionals

`afx.egemaps(signal, sr)` returns `{name: value}` in the order of `afx.EGEMAPS_NAMES`, which is openSMILE's, so tables from either tool line up by column name. Statistics over voiced frames are 0 for a clip without voicing, as in openSMILE.

In [ ]:
values = pd.Series(afx.egemaps(x, SR))
assert list(values.index) == list(afx.EGEMAPS_NAMES) and len(values) == 88
print(len(values), 'functionals:', ', '.join(values.index[:3]), '...', values.index[-1])
values.loc[['F0semitoneFrom27.5Hz_sma3nz_amean', 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
            'jitterLocal_sma3nz_amean', 'shimmerLocaldB_sma3nz_amean', 'HNRdBACF_sma3nz_amean',
            'F1frequency_sma3nz_amean', 'F2frequency_sma3nz_amean', 'loudness_sma3_amean',
            'VoicedSegmentsPerSec', 'MeanVoicedSegmentLengthSec', 'equivalentSoundLevel_dBp']].round(4)

The synthesis used F1 = 730 Hz and F2 = 1090 Hz (vowel /a/), and one voiced stretch of 1.5 s.

## The 25 LLDs over time

`egemaps_lld` is a framed feature like any other: 25 rows (`afx.EGEMAPS_LLD_NAMES`) every 10 ms, through a config with `preserve_shape`. The 15 voice descriptors from F0 onwards are NaN on unvoiced frames, where openSMILE writes 0.

In [ ]:
out = afx.extract_all_features(x, SR, {'features': ['egemaps_lld'], 'preserve_shape': True}, return_metadata=True)
lld = pd.DataFrame(out['features']['egemaps_lld'].T, columns=afx.EGEMAPS_LLD_NAMES,
                   index=pd.Index(out['metadata']['egemaps_lld']['times'], name='time (s)'))
print(lld.shape, '| voiced frames:', int(lld['F0semitoneFrom27.5Hz_sma3nz'].notna().sum()))
print('NaN fraction per row:', lld.isna().mean().round(2).to_dict())

In [ ]:
t = lld.index.to_numpy()
glide = np.where((t >= 0.25) & (t <= 1.75), 110 + 70 * (t - 0.25) / 1.5, np.nan)
f0_hz = 27.5 * 2 ** (lld['F0semitoneFrom27.5Hz_sma3nz'] / 12)

fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
axes[0].plot(t, lld['Loudness_sma3'])
axes[0].set_ylabel('loudness')
axes[1].plot(t, glide, 'k--', lw=1, label='synthesized')
axes[1].plot(t, f0_hz, label='F0 (eGeMAPS)')
axes[1].set_ylabel('Hz')
axes[1].legend(loc='upper left')
axes[2].plot(t, lld['HNRdBACF_sma3nz'])
axes[2].set_ylabel('HNR (dB)')
axes[3].plot(t, 100 * lld['jitterLocal_sma3nz'], label='jitter (%)')
axes[3].plot(t, lld['shimmerLocaldB_sma3nz'], label='shimmer (dB)')
axes[3].legend(loc='upper left')
axes[3].set_xlabel('Time (s)')
fig.suptitle('eGeMAPS LLDs; voice descriptors are NaN outside the voiced stretch')
fig.tight_layout()

## Recovering F0, jitter and shimmer


A grid of constant-F0 vowels /i/ and /u/ at three pitches with increasing jitter and shimmer, 2 s each. The table compares the functionals with the realized values.

In [ ]:
rows = []
for vowel in ('i', 'u'):
    for f0 in (100.0, 160.0, 240.0):
        for jitter, shimmer in ((0.0, 0.0), (0.005, 0.025), (0.01, 0.05), (0.02, 0.1)):
            y, true = demo_data.synthetic_voice(f0=f0, jitter=jitter, shimmer=shimmer, vowel=vowel, seed=int(f0),
                                                return_truth=True)
            v = afx.egemaps(y, SR)
            rows.append({
                'vowel': vowel, 'f0_hz': f0, 'jitter_param': jitter, 'shimmer_param': shimmer,
                'f0_st_true': true['f0_semitones'], 'f0_st': v['F0semitoneFrom27.5Hz_sma3nz_amean'],
                'jitter_true': true['jitter'], 'jitter': v['jitterLocal_sma3nz_amean'],
                'shimmer_true': true['shimmer_db'], 'shimmer': v['shimmerLocaldB_sma3nz_amean'],
            })
grid = pd.DataFrame(rows)
grid['f0_error_st'] = grid['f0_st'] - grid['f0_st_true']
print(f"F0 over {len(grid)} clips: max |error| {grid['f0_error_st'].abs().max():.3f} semitones")
grid.groupby(['vowel', 'jitter_param', 'shimmer_param'])[
    ['jitter_true', 'jitter', 'shimmer_true', 'shimmer']].mean().round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, name, unit in ((axes[0], 'jitter', '(ratio)'), (axes[1], 'shimmer', '(dB)')):
    for (vowel, f0), part in grid.groupby(['vowel', 'f0_hz']):
        ax.plot(part[f'{name}_true'], part[name], 'o-' if vowel == 'i' else 's--', label=f'/{vowel}/ {f0:.0f} Hz')
    top = grid[[f'{name}_true', name]].to_numpy().max() * 1.05
    ax.plot([0, top], [0, top], 'k:', lw=1, label='identity')
    ax.set_xlabel(f'realized {name} {unit}')
    ax.set_ylabel(f'eGeMAPS {name} {unit}')
axes[1].legend(fontsize=7)
fig.tight_layout()

perturbed = grid[grid['jitter_param'] > 0]
ratios = perturbed.assign(jitter_ratio=perturbed['jitter'] / perturbed['jitter_true'],
                          shimmer_ratio=perturbed['shimmer'] / perturbed['shimmer_true'])
print('eGeMAPS / realized, over vowels and pitches:')
print(ratios[['jitter_ratio', 'shimmer_ratio']].describe().loc[['min', '50%', 'max']].round(2))
print('shimmer at zero shimmer (dB):', grid.loc[grid['shimmer_param'] == 0, 'shimmer'].round(2).tolist())

F0 is within 0.07 semitones of the synthesis on every clip. Jitter and shimmer rise with the realized values at each vowel and pitch, but the scale depends on both: measured over realized jitter ranges from about 0.5 to 1.4 (median 0.9), and shimmer has a floor of 0.15 to 0.3 dB at zero shimmer from the additive noise. Compare them across clips of similar voices within one dataset and one tool; values from different tools are not interchangeable (openSMILE's are in the last cell).

## The `egemaps` preset in a pipeline

`egemaps@1` is a preset with the single clip-level feature `egemaps`, so `AFXFeatureExtractor(config='egemaps@1')` returns the 88 functionals per clip. Task: tell modal voices (jitter 0.3 %, shimmer 2 %) from rough ones (jitter 2 %, shimmer 10 %) at random F0, vowel and SNR.

In [ ]:
rng = np.random.default_rng(0)
voices, labels = [], []
for i in range(32):
    rough = i % 2 == 1
    voices.append(demo_data.synthetic_voice(
        f0=float(rng.uniform(90, 250)), jitter=0.02 if rough else 0.003, shimmer=0.1 if rough else 0.02,
        vowel='aiu'[i % 3], seconds=1.5, voiced=(0.2, 1.3), snr_db=float(rng.uniform(20, 40)), seed=100 + i))
    labels.append('rough' if rough else 'modal')
labels = np.array(labels)

pipe = make_pipeline(AFXFeatureExtractor(config='egemaps@1'), StandardScaler(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
scores = cross_val_score(pipe, voices, labels, cv=cv)
print(f'4-fold accuracy {scores.mean():.3f} +/- {scores.std():.3f} (chance 0.5)')

Output columns are `egemaps_0` to `egemaps_87`, in the order of `afx.EGEMAPS_NAMES`; map them back to read the model. Rough voices also roughen the F0 contour and the spectral flux, and the model weighs those as much as jitter and shimmer.

In [ ]:
pipe.fit(voices, labels)
extractor = pipe[0]
print(extractor.get_feature_names_out()[:3], extractor.provenance_['preset_id'])
coef = pd.Series(pipe[-1].coef_[0], index=afx.EGEMAPS_NAMES)
coef.reindex(coef.abs().sort_values(ascending=False).index)[:8].round(3)

## openSMILE comparison (optional)

openSMILE's licence allows research use only, so it is not an AFX dependency. With `pip install opensmile`, this cell compares a few functionals on the grid's modal and rough settings; without it the cell prints a note and the notebook still completes. `docs/egemaps.md` has all 88 over a 20-clip corpus.

In [ ]:
if importlib.util.find_spec('opensmile') is None:
    print('opensmile is not installed; comparison skipped')
else:
    import opensmile

    smile = opensmile.Smile(feature_set=opensmile.FeatureSet.eGeMAPSv02,
                            feature_level=opensmile.FeatureLevel.Functionals)
    compare = ['F0semitoneFrom27.5Hz_sma3nz_amean', 'jitterLocal_sma3nz_amean', 'shimmerLocaldB_sma3nz_amean',
               'HNRdBACF_sma3nz_amean', 'loudness_sma3_amean', 'F1frequency_sma3nz_amean',
               'equivalentSoundLevel_dBp']
    table = []
    for jitter, shimmer in ((0.0, 0.0), (0.01, 0.05), (0.02, 0.1)):
        y, true = demo_data.synthetic_voice(f0=140.0, jitter=jitter, shimmer=shimmer, seed=1, return_truth=True)
        ours = afx.egemaps(y, SR)
        ref = smile.process_signal(y, SR).iloc[0]
        for name in compare:
            table.append({'jitter, shimmer': f'{jitter}, {shimmer}', 'functional': name,
                          'afx': ours[name], 'opensmile': float(ref[name])})
    print('realized jitter and shimmer of the last clip:', round(true['jitter'], 4), round(true['shimmer_db'], 3))
    display(pd.DataFrame(table).pivot(index='functional', columns='jitter, shimmer',
                                       values=['afx', 'opensmile']).round(4))